# Volatility-Adjusted Momentum — 12M — Winner Drift

One signal, one formation horizon and one maintenance method. The experiment contains nine cells: rebalance every 1, 3 or 6 months × target N=12, 24 or 50.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))
from momentum_india.notebook_views import ResearchNotebook

research = ResearchNotebook('vol_adjusted', '12M', 'winner_drift')

## 1. Signal and portfolio rule

Score = formation-period price return / annualized sample standard deviation of daily log returns. Annualization uses √252. The signal rewards stronger returns per unit of realized variability; it does not set inverse-volatility portfolio weights.

Continuing holdings retain their naturally drifted weights, capped at 2/N at scheduled rebalances. Exits and entrants are paired in stable symbol order; an entrant receives min(exit weight, 1/N). Excess exit weight and cap trims are spread equally among continuing holdings with room below the cap. Unmatched entrants share existing cash up to 1/N each. Residual cash is retained. The cap is a scheduled target constraint, not a daily trim rule.

## 2. Universe → ranking → actual portfolio

The example uses the latest monthly N=24 signal and exposes the ranking inputs and actual target weights.

In [2]:
research.snapshot()

Symbol,Return / volatility,MDTV (INR)
NSE:CUPID-EQ,11.545,"2,261,224,535.95"
NSE:SBC-EQ,5.605,"370,044,564.64"
NSE:ATHERENERG-EQ,5.211,"2,745,338,022.88"
NSE:RPTECH-EQ,4.773,"78,147,051.55"
NSE:SANSERA-EQ,3.834,"511,549,310.60"
NSE:LAURUSLABS-EQ,3.659,"2,123,882,567.10"
NSE:HFCL-EQ,3.254,"3,688,024,397.99"
NSE:KMEW-EQ,3.210,"175,654,741.20"
NSE:SILVERTUC-EQ,3.098,"65,229,764.81"
NSE:ACUTAAS-EQ,3.094,"1,065,873,412.85"


Symbol,Leg,Actual weight,Current selection,Execution status,Return / volatility,MDTV (INR)
NSE:CUPID-EQ,long,2.2%,True,selected,11.545,"2,261,224,535.95"
NSE:SBC-EQ,long,0.5%,True,selected,5.605,"370,044,564.64"
NSE:ATHERENERG-EQ,long,0.7%,True,selected,5.211,"2,745,338,022.88"
NSE:RPTECH-EQ,long,0.3%,True,selected,4.773,"78,147,051.55"
NSE:SANSERA-EQ,long,0.7%,True,selected,3.834,"511,549,310.60"
NSE:LAURUSLABS-EQ,long,0.9%,True,selected,3.659,"2,123,882,567.10"
NSE:HFCL-EQ,long,0.5%,True,selected,3.254,"3,688,024,397.99"
NSE:KMEW-EQ,long,0.4%,True,selected,3.210,"175,654,741.20"
NSE:SILVERTUC-EQ,long,0.3%,True,selected,3.098,"65,229,764.81"
NSE:ACUTAAS-EQ,long,0.3%,True,selected,3.094,"1,065,873,412.85"


Symbol,Formation start,Start adjusted close,Signal close date,End adjusted close,Formation price return
NSE:CUPID-EQ,2025-07-31,30.18,2026-07-31,230.62,664.2%


## 3. Return layers across all nine cells

Raw is before trading charges. After-cost gross/pre-tax deducts modeled trading charges. The long-only post-tax overlay additionally applies the annual equity-gains ledger. The academic reference instead compares raw and borrowing-adjusted layers.

In [3]:
research.layers_bridge()

Rebalance,First date,Last date,Sessions
1M,2007-05-03,2026-08-28,4771
3M,2007-07-02,2026-08-28,4729
6M,2007-07-02,2026-08-28,4729


Rebalance,N,Raw,After costs,Post-tax overlay
1M,12,11.5%,11.2%,10.3%
1M,24,12.7%,12.4%,11.5%
1M,50,14.0%,13.5%,12.7%
3M,12,13.8%,13.5%,12.0%
3M,24,13.6%,13.1%,12.0%
3M,50,12.0%,11.6%,10.9%
6M,12,7.9%,7.7%,7.2%
6M,24,10.8%,10.6%,9.6%
6M,50,10.4%,10.0%,9.3%


## 4. Risk-adjusted results

Sharpe uses daily excess returns relative to the liquid fund. VaR and expected shortfall are historical monthly 95% loss measures. Partial first/last months are included. Time below prior peak counts days awaiting a new all-time high—not losing days. The initial invested capital is included as the first peak.

In [4]:
research.risk_grid()

Rebalance,N,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,12,10.3%,0.270,-59.4%,7.0%
1M,24,11.5%,0.342,-68.1%,6.9%
1M,50,12.7%,0.438,-69.0%,6.1%
3M,12,12.0%,0.336,-63.8%,8.0%
3M,24,12.0%,0.330,-69.9%,8.3%
3M,50,10.9%,0.310,-70.3%,6.9%
6M,12,7.2%,0.072,-60.7%,5.4%
6M,24,9.6%,0.219,-69.2%,8.6%
6M,50,9.3%,0.205,-73.6%,8.7%


Rebalance,N,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,12,11.2%,91.5%,1594
1M,24,12.6%,90.5%,1593
1M,50,12.4%,88.4%,1592
3M,12,13.6%,93.8%,1567
3M,24,15.2%,93.3%,1593
3M,50,13.4%,91.3%,1602
6M,12,11.3%,93.1%,1780
6M,24,15.8%,94.3%,1639
6M,50,15.8%,93.4%,1643


### CAGR

In [5]:
research.heatmap('cagr')

### Sharpe ratio

In [6]:
research.heatmap('sharpe')

### Maximum drawdown

In [7]:
research.heatmap('maximum_drawdown')

### Monthly 95% VaR

In [8]:
research.heatmap('monthly_var_95')

## 5. Equity paths and matched risks

Each chart fixes breadth and compares rebalance frequencies. Final-layer curves and the price benchmark start visible; other return layers remain in the selectable legend. Logarithmic axes make early and late periods comparable; the bottom range slider preserves the full history. Curves display weekly observations for readability, while every statistic uses the complete daily series.

### N=12

In [9]:
research.equity(12)

Rebalance,Return layer,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,Raw,11.5%,0.346,-58.3%,6.7%
1M,After costs,11.2%,0.323,-58.7%,7.0%
1M,Post-tax overlay,10.3%,0.270,-59.4%,7.0%
1M,Nifty 50 price,9.7%,0.219,-59.9%,6.9%
3M,Raw,13.8%,0.425,-62.7%,7.9%
3M,After costs,13.5%,0.408,-62.9%,8.0%
3M,Post-tax overlay,12.0%,0.336,-63.8%,8.0%
3M,Nifty 50 price,9.5%,0.209,-59.9%,7.0%
6M,Raw,7.9%,0.116,-59.1%,5.3%
6M,After costs,7.7%,0.105,-59.2%,5.4%


Rebalance,Return layer,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,Raw,10.9%,90.1%,1581
1M,After costs,11.1%,90.6%,1588
3M,Raw,13.4%,92.9%,1518
3M,After costs,13.5%,93.2%,1550
6M,Raw,10.9%,92.1%,1684
6M,After costs,11.0%,92.3%,1733
1M,Post-tax overlay,11.2%,91.5%,1594
3M,Post-tax overlay,13.6%,93.8%,1567
6M,Post-tax overlay,11.3%,93.1%,1780
1M,Nifty 50 price,13.1%,92.4%,1520


### N=24

In [10]:
research.equity(24)

Rebalance,Return layer,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,Raw,12.7%,0.415,-67.5%,6.6%
1M,After costs,12.4%,0.394,-67.8%,6.9%
1M,Post-tax overlay,11.5%,0.342,-68.1%,6.9%
1M,Nifty 50 price,9.7%,0.219,-59.9%,6.9%
3M,Raw,13.6%,0.408,-68.8%,8.2%
3M,After costs,13.1%,0.385,-69.1%,8.3%
3M,Post-tax overlay,12.0%,0.330,-69.9%,8.3%
3M,Nifty 50 price,9.5%,0.209,-59.9%,7.0%
6M,Raw,10.8%,0.280,-68.1%,8.5%
6M,After costs,10.6%,0.268,-68.3%,8.6%


Rebalance,Return layer,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,Raw,12.4%,89.1%,1581
1M,After costs,12.5%,89.7%,1589
3M,Raw,15.0%,92.5%,1581
3M,After costs,15.0%,92.7%,1585
6M,Raw,15.5%,93.6%,1612
6M,After costs,15.6%,93.7%,1633
1M,Post-tax overlay,12.6%,90.5%,1593
3M,Post-tax overlay,15.2%,93.3%,1593
6M,Post-tax overlay,15.8%,94.3%,1639
1M,Nifty 50 price,13.1%,92.4%,1520


### N=50

In [11]:
research.equity(50)

Rebalance,Return layer,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,Raw,14.0%,0.527,-68.2%,6.0%
1M,After costs,13.5%,0.493,-68.5%,6.1%
1M,Post-tax overlay,12.7%,0.438,-69.0%,6.1%
1M,Nifty 50 price,9.7%,0.219,-59.9%,6.9%
3M,Raw,12.0%,0.376,-69.5%,6.9%
3M,After costs,11.6%,0.352,-69.6%,6.9%
3M,Post-tax overlay,10.9%,0.310,-70.3%,6.9%
3M,Nifty 50 price,9.5%,0.209,-59.9%,7.0%
6M,Raw,10.4%,0.261,-72.7%,8.6%
6M,After costs,10.0%,0.241,-72.8%,8.7%


Rebalance,Return layer,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,Raw,12.2%,86.8%,1578
1M,After costs,12.3%,87.4%,1588
3M,Raw,13.2%,90.0%,1594
3M,After costs,13.2%,90.4%,1599
6M,Raw,15.5%,92.9%,1637
6M,After costs,15.6%,93.0%,1641
1M,Post-tax overlay,12.4%,88.4%,1592
3M,Post-tax overlay,13.4%,91.3%,1602
6M,Post-tax overlay,15.8%,93.4%,1643
1M,Nifty 50 price,13.1%,92.4%,1520


## 6. Recovery burden

The longest underwater episode is shown by its peak, trough and recovery dates. Unrecovered episodes remain explicitly open.

In [12]:
research.recovery()

Series,Peak,Trough,Recovery,Sessions,Calendar days,Episode loss
1M,2008-01-07,2008-12-05,2014-06-24,1593,2359,-68.1%
3M,2008-01-07,2008-11-20,2014-06-24,1593,2359,-69.9%
6M,2008-01-07,2008-10-27,2014-09-01,1639,2428,-69.2%
Nifty · 1M,2008-01-08,2008-10-27,2014-03-06,1520,2248,-59.9%
Nifty · 3M,2008-01-08,2008-10-27,2014-03-06,1520,2248,-59.9%
Nifty · 6M,2008-01-08,2008-10-27,2014-03-06,1520,2248,-59.9%


## 7. Portfolio behavior

Scheduled turnover is (buy value + sell value)/(2 × pre-trade equity). Retention compares successive scheduled target name sets. Cash and total charges include the intervening daily path.

In [13]:
research.behavior()

Rebalance,N,Mean names at rebalance,Mean cash weight,Scheduled name retention
1M,12,15.34,45.8%,68.7%
1M,24,31.35,35.2%,72.2%
1M,50,72.39,31.7%,78.2%
3M,12,15.60,25.3%,49.1%
3M,24,31.91,10.8%,53.6%
3M,50,73.51,23.3%,62.8%
6M,12,16.79,50.4%,37.9%
6M,24,32.10,12.7%,38.6%
6M,50,66.44,8.2%,43.6%


Rebalance,N,Mean scheduled turnover,All trading charges (INR),Days with stop sales
1M,12,15.5%,"2,093,870.11",0
1M,24,15.5%,"2,399,808.32",0
1M,50,13.3%,"2,467,591.97",0
3M,12,34.3%,"2,383,646.91",0
3M,24,38.4%,"2,338,885.56",0
3M,50,26.9%,"1,356,710.85",0
6M,12,27.9%,"450,723.50",0
6M,24,53.8%,"1,223,329.29",0
6M,50,50.3%,"1,055,028.17",0


## 8. Market-state attribution

This is an observation, not an extra strategy filter. Prior-close Nifty 50 versus SMA(200) defines up/down; 63-session volatility versus its expanding historical median defines high/low volatility. The representative monthly N=24 path is shown with shaded states.

In [14]:
research.regimes()

Market state,Sessions,Mean daily return,Daily volatility,Positive days
Down / High volatility,748,-0.06%,1.17%,55.35%
Down / Low volatility,609,0.03%,0.86%,55.83%
Up / High volatility,720,0.15%,1.43%,60.83%
Up / Low volatility,2694,0.05%,0.75%,59.06%


## 9. Complete portfolio and trade evidence

Separate CSV files retain all scheduled portfolios, actual trades and risk layers for this exact signal/lookback/maintenance combination.

In [15]:
research.portfolio_exports()

Rebalance,N,First rebalance,Last rebalance,Rebalance dates,Holding rows
1M,12,2007-05-03,2026-08-03,232,3560
1M,24,2007-05-03,2026-08-03,232,7273
1M,50,2007-05-03,2026-08-03,232,16794
3M,12,2007-07-02,2026-07-01,77,1201
3M,24,2007-07-02,2026-07-01,77,2457
3M,50,2007-07-02,2026-07-01,77,5660
6M,12,2007-07-02,2026-07-01,39,655
6M,24,2007-07-02,2026-07-01,39,1252
6M,50,2007-07-02,2026-07-01,39,2591


## Findings

In [16]:
research.conclusion()

Matched reference: [Raw Momentum — 12M — Winner Drift](05_raw_momentum_12m_winner_drift.ipynb). The comparison holds formation, maintenance, frequency and N fixed.